# 02 Classification: High Risk Flag


## Objective

??????????????????????????? `high_risk_flag`?


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().resolve().parent if (Path.cwd().resolve().parent / "src").exists() else PROJECT_ROOT

sys.path.insert(0, str(PROJECT_ROOT / "src"))
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

from config import (
    CLASSIFICATION_TARGET,
    FIGURES_DIR,
    HIGH_RISK_LEAKAGE_COLUMNS,
    ID_COLUMNS,
    RANDOM_STATE,
    RESULTS_DIR,
)
from data_utils import ensure_project_dirs, load_processed_dataset
from feature_engineering import get_excluded_columns, make_feature_target
from model_utils import run_classification_experiment
from visualization import plot_confusion_matrix, plot_feature_importance, plot_roc_curve

np.random.seed(RANDOM_STATE)
ensure_project_dirs()


PROJECT_ROOT = C:\Users\qintian\Desktop\大数据\final_report_project


## Leakage-Aware Feature Check

???????????????????????????


In [2]:
df = load_processed_dataset(fallback_to_raw=True)
excluded = get_excluded_columns(task="classification")
reasons = []
for column in excluded:
    if column == CLASSIFICATION_TARGET:
        reasons.append("classification target")
    elif column in HIGH_RISK_LEAKAGE_COLUMNS:
        reasons.append("specified leakage/outcome column")
    elif column in ID_COLUMNS:
        reasons.append("identifier")
    else:
        reasons.append("excluded by configuration")

exclusion_table = pd.DataFrame({"excluded_column": excluded, "reason": reasons})
exclusion_table.to_csv(RESULTS_DIR / "classification_feature_exclusion.csv", index=False)

X, y = make_feature_target(df, task="classification")
print(f"Feature matrix shape: {X.shape}")
print(f"Target positive rate: {y.mean():.4f}")
display(exclusion_table)


Feature matrix shape: (3500, 22)
Target positive rate: 0.2014


,excluded_column,reason
0,anxiety_score,specified leakage/outcome column
1,depression_score,specified leakage/outcome column
2,digital_dependence_score,specified leakage/outcome column
3,focus_score,specified leakage/outcome column
4,happiness_score,specified leakage/outcome column
5,high_risk_flag,classification target
6,id,identifier
7,productivity_score,specified leakage/outcome column
8,stress_level,specified leakage/outcome column


## Train and Evaluate Models

???? CPU ?? scikit-learn ?????????? `random_state=42`?


In [3]:
classification_result = run_classification_experiment(df)
classification_metrics = classification_result["metrics"]
print(f"Best model: {classification_result['best_model_name']}")
display(classification_metrics)


Best model: random_forest


,model,accuracy,precision,recall,f1,roc_auc,n_train,n_test,feature_count
0,random_forest,0.810286,0.583333,0.198864,0.296610,0.763623,2625,875,22
1,logistic_regression,0.742857,0.406844,0.607955,0.487472,0.745440,2625,875,22
2,gradient_boosting,0.826286,0.653846,0.289773,0.401575,0.736206,2625,875,22


## Classification Figures


In [4]:
plot_confusion_matrix(
    classification_result["confusion_matrix"],
    FIGURES_DIR / "classification_best_confusion_matrix.png",
)
if classification_result["y_score"] is not None:
    plot_roc_curve(
        classification_result["y_test"],
        classification_result["y_score"],
        FIGURES_DIR / "classification_best_roc_curve.png",
    )
plot_feature_importance(
    classification_result["feature_importance"],
    FIGURES_DIR / "classification_permutation_importance.png",
    title="Classification Permutation Importance",
)
classification_result["feature_importance"].head(15)


,feature,importance_mean,importance_std
0,sleep_hours,0.034776,0.011157
1,device_hours_per_day,0.028936,0.008868
2,device_to_sleep_ratio,0.015569,0.009509
3,phone_unlocks,0.009327,0.003307
4,social_to_study_ratio,0.005252,0.002079
5,income_level,0.004900,0.001880
6,gender,0.004669,0.003336
7,daily_role,0.004268,0.002107
8,activity_sleep_interaction,0.004160,0.005014
9,notifications_per_device_hour,0.003998,0.002408


<Figure size 800x640 with 0 Axes>

## Notes for Report

????????? `results/classification_high_risk_metrics.csv` ???????????????
